In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
pd.set_option('display.max_columns', None)

In [ ]:
!pip install pandas pyarrow -q
# import venue_enrichment

In [ ]:
#!/usr/bin/env python3
"""
venue_enrichment.py  —  Google Colab
====================================
Attach authoritative VENUE attributes for Paper 3's visibility-penalty analysis
by joining two static dumps on the ISSNs you hold (oa_issn), with a JOURNAL-NAME
fallback (oa_journal) for records whose ISSN doesn't match:

  DOAJ public dump   -> is the venue in DOAJ (fully OA), does it charge an APC.
  Scimago (SJR) table-> journal SJR, best quartile (Q1-Q4), and journal country.

Why these matter for Paper 3:
  - OA status is a CONFOUNDER in the citation-penalty matching (LICs have the
    highest OA rates; OA lifts citations). oa_oa_status is OpenAlex's guess;
    DOAJ is the reference for gold-OA journals + whether an APC applies.
  - Journal quartile / SJR is the VENUE-IMPACT outcome itself: do locally-led
    papers land in lower-ranked journals than externally-led ones?
  - Journal country lets you flag whether African-led work is published in
    African-domiciled vs Northern-domiciled venues.

MATCHING: ISSN is primary (exact). Journal-name is a fallback, used only when a
paper has no ISSN match. Name matching is exact-after-normalization, so it will
miss abbreviated titles ("J. Med. Entomol.") and carries a small false-positive
risk (distinct journals sharing a title). The venue_matched_by column flags each
row as 'issn' / 'name' / '' so you can report ISSN-only as the strict version and
name-matched as a sensitivity layer.

APPROACH: local join, no per-record API crawl.
  DOAJ dump:    https://doaj.org/csv           (journalcsv..., updated daily)
  Scimago dump: https://www.scimagojr.com/journalrank.php  ->  "Download data"
                (a ';'-separated CSV; grab the latest year, or one file per year
                 if you want time-varying quartiles).

INPUT: your latest merged parquet (must have oa_issn and oa_journal).
SETUP:  !pip install pandas pyarrow -q
        # upload the two CSVs to Drive and set the paths below.
"""

import re, sys
import pandas as pd

# =========================================================================== #
# CONFIG                                                                        #
# =========================================================================== #
IN_PARQUET  = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/llm_screen/screened_llm.parquet"
OUT_PARQUET = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/with_venue.parquet"
DOAJ_CSV    = "/content/drive/MyDrive/COGNOSX/ticks/external_data/doaj_journalcsv_20260630_2320_utf8.csv"       # from doaj.org/csv
SCIMAGO_CSV = "/content/drive/MyDrive/COGNOSX/ticks/external_data/scimagojr_2025.csv"      # ';'-separated


# =========================================================================== #
# NA-safe helpers                                                              #
# =========================================================================== #
def S(x):
    """NA/None-safe string coercion (parquet stores missing as pd.NA)."""
    return "" if x is None or (isinstance(x, float) and pd.isna(x)) or x is pd.NA else str(x)

def norm_issn(x):
    """'0168-8162' / '01688162' / '0168-8162 (Print)' -> '01688162' (8 chars, X upper)."""
    s = re.sub(r"[^0-9xX]", "", S(x)).upper()
    return s if len(s) == 8 else ""

def issn_set(cell):
    """oa_issn may hold several ISSNs joined by | , ; / space — return all normalized."""
    return {n for part in re.split(r"[|,;/ ]+", S(cell)) if (n := norm_issn(part))}

def norm_journal(x):
    """lowercase, drop leading 'the', & -> and, strip punctuation, collapse spaces."""
    s = S(x).lower().replace("&", " and ")
    s = re.sub(r"^the\s+", "", s)
    s = re.sub(r"[^a-z0-9 ]", " ", s)
    return re.sub(r"\s+", " ", s).strip()


# =========================================================================== #
# 1. DOAJ lookups:  issn -> {in_doaj, has_apc}   and   name -> {...}           #
# =========================================================================== #
def _doaj_cols(d):
    cols = {c.lower(): c for c in d.columns}
    def find(*cands):
        for c in cands:
            if c in cols: return cols[c]
        return None
    c_pissn = find("journal issn (print version)", "issn", "pissn")
    c_eissn = find("journal eissn (online version)", "eissn")
    c_apc   = find("apc", "does the journal charge an apc", "apc charged")
    c_title = find("journal title", "title")
    return c_pissn, c_eissn, c_apc, c_title

def load_doaj(path):
    d = pd.read_csv(path, dtype=str, on_bad_lines="skip")
    c_pissn, c_eissn, c_apc, _ = _doaj_cols(d)
    lookup = {}
    for _, r in d.iterrows():
        has_apc = S(r.get(c_apc)).strip().lower() in ("yes", "true", "1")
        for col in (c_pissn, c_eissn):
            if col:
                n = norm_issn(r.get(col))
                if n:
                    lookup[n] = {"in_doaj": True, "has_apc": has_apc}
    print(f"[doaj] {len(lookup):,} ISSNs", file=sys.stderr)
    return lookup

def load_doaj_names(path):
    d = pd.read_csv(path, dtype=str, on_bad_lines="skip")
    _, _, c_apc, c_title = _doaj_cols(d)
    look = {}
    for _, r in d.iterrows():
        nm = norm_journal(r.get(c_title))
        if nm:
            has_apc = S(r.get(c_apc)).strip().lower() in ("yes", "true", "1")
            look[nm] = {"in_doaj": True, "has_apc": has_apc}
    print(f"[doaj] {len(look):,} journal names", file=sys.stderr)
    return look


# =========================================================================== #
# 2. Scimago lookups:  issn -> {...}   and   name -> {...}                     #
# =========================================================================== #
def _scimago_cols(s):
    cols = {c.lower(): c for c in s.columns}
    def find(*cands):
        for c in cands:
            if c in cols: return cols[c]
        return None
    return (find("issn"), find("sjr"),
            find("sjr best quartile", "best quartile", "quartile"),
            find("country"), find("title"))

def _sjr_float(x):
    try:
        return float(S(x).replace(",", "."))
    except ValueError:
        return None

def load_scimago(path):
    # Scimago exports are ';'-separated and use ',' as the decimal mark
    s = pd.read_csv(path, sep=";", dtype=str, on_bad_lines="skip")
    c_issn, c_sjr, c_q, c_ctry, _ = _scimago_cols(s)
    lookup = {}
    for _, r in s.iterrows():
        rec = {"sjr": _sjr_float(r.get(c_sjr)),
               "quartile": S(r.get(c_q)).strip(),
               "journal_country": S(r.get(c_ctry)).strip()}
        for n in issn_set(r.get(c_issn)):   # Scimago packs multiple ISSNs in one field
            lookup[n] = rec
    print(f"[scimago] {len(lookup):,} ISSNs", file=sys.stderr)
    return lookup

def load_scimago_names(path):
    s = pd.read_csv(path, sep=";", dtype=str, on_bad_lines="skip")
    _, c_sjr, c_q, c_ctry, c_title = _scimago_cols(s)
    look = {}
    for _, r in s.iterrows():
        nm = norm_journal(r.get(c_title))
        if nm:
            look[nm] = {"sjr": _sjr_float(r.get(c_sjr)),
                        "quartile": S(r.get(c_q)).strip(),
                        "journal_country": S(r.get(c_ctry)).strip()}
    print(f"[scimago] {len(look):,} journal names", file=sys.stderr)
    return look


# =========================================================================== #
# 3. join onto the corpus  (ISSN primary, journal-name fallback)              #
# =========================================================================== #
def enrich(df, doaj, scimago, doaj_nm, scimago_nm):
    def row_venue(row):
        issns = issn_set(row.get("oa_issn"))
        # --- primary: ISSN (exact) ---
        in_doaj = any(doaj.get(i, {}).get("in_doaj") for i in issns)
        has_apc = any(doaj.get(i, {}).get("has_apc") for i in issns)
        sci = next((scimago[i] for i in issns if i in scimago), None)
        matched_by = "issn" if (issns and (in_doaj or sci)) else ""
        # --- fallback: journal name (only if ISSN found nothing) ---
        if not matched_by:
            nm = norm_journal(row.get("oa_journal"))
            if nm:
                dn = doaj_nm.get(nm)
                if dn:
                    in_doaj, has_apc = dn["in_doaj"], dn["has_apc"]
                sci = scimago_nm.get(nm)
                if dn or sci:
                    matched_by = "name"
        return pd.Series({
            "venue_in_doaj": bool(in_doaj),
            "venue_doaj_apc": bool(has_apc),
            "venue_sjr": sci["sjr"] if sci else None,
            "venue_quartile": sci["quartile"] if sci else "",
            "venue_country": sci["journal_country"] if sci else "",
            "venue_matched_by": matched_by,
        })
    ven = df.apply(row_venue, axis=1)
    out = pd.concat([df, ven], axis=1)
    out.to_parquet(OUT_PARQUET, index=False)
    print(f"[write] {OUT_PARQUET}", file=sys.stderr)

    by = out["venue_matched_by"].value_counts()
    q_cov = (out["venue_quartile"] != "").mean()
    print(f"[coverage] matched by ISSN: {by.get('issn', 0):,}  "
          f"by name: {by.get('name', 0):,}  "
          f"unmatched: {(out['venue_matched_by'] == '').sum():,}  "
          f"total: {len(out):,}  |  has quartile: {q_cov:.1%}", file=sys.stderr)
    return out


# =========================================================================== #
# 4. quick Paper-3 read-out: venue impact by leadership                       #
# =========================================================================== #
def venue_by_leadership(out):
    """Descriptive: are locally-led papers in lower quartiles / less OA?
    Uses the _filled country columns from the ROR step if present."""
    def to_set(s, sep=";"):
        return {x for x in S(s).split(sep) if x}
    fc = ("first_author_countries_filled"
          if "first_author_countries_filled" in out.columns
          else "first_author_countries")
    rows = []
    for _, r in out.iterrows():
        target = to_set(r.get("study_countries"))
        if not target or not (to_set(r.get("study_bloc")) & {"ssa", "maghreb"}):
            continue
        led = "local-led" if (to_set(r.get(fc)) & target) else "external-led"
        rows.append({"led": led, "quartile": S(r.get("venue_quartile")),
                     "in_doaj": bool(r.get("venue_in_doaj")), "sjr": r.get("venue_sjr")})
    t = pd.DataFrame(rows)
    if t.empty:
        print("[readout] no bloc rows with study country + venue"); return
    print("\n[readout] SSA+Maghreb papers, venue by leadership:")
    print(t.groupby("led").agg(
        n=("led", "size"),
        pct_Q1=("quartile", lambda s: round((s == "Q1").mean(), 3)),
        pct_in_doaj=("in_doaj", "mean"),
        median_sjr=("sjr", "median")).to_string())


# =========================================================================== #
if __name__ == "__main__":
    df_all = pd.read_parquet(IN_PARQUET)
    df = df_all[df_all["llm_label"].notna()]
    print(f"[in] {len(df):,} rows", file=sys.stderr)
    doaj = load_doaj(DOAJ_CSV)
    scimago = load_scimago(SCIMAGO_CSV)
    doaj_nm = load_doaj_names(DOAJ_CSV)
    scimago_nm = load_scimago_names(SCIMAGO_CSV)
    out = enrich(df, doaj, scimago, doaj_nm, scimago_nm)
    venue_by_leadership(out)

In [ ]:
out